# DATA CLEANING

In [ ]:
import pandas as pd
import numpy as np
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

# Download NLTK assets (only first time)
nltk.download('stopwords')
nltk.download('wordnet')

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Unzipping corpora/stopwords.zip.
[nltk_data] Downloading package wordnet to /root/nltk_data...


True

In [ ]:
df = pd.read_excel("/content/Tickets Data.xlsx")

In [ ]:
df.head()

,subject,Body,type,queue,priority,Tags
0,subject,"dear support team, i would like to report a se...",Incident,Technical Support,high,"security,outage,disruption,data breach"
1,major security incident,"dear customer support team,\n\ni am writing to...",Incident,Technical Support,high,"account,disruption,outage,it,tech support"
2,account disruption,"dear customer support team,\n\ni hope this mes...",Request,Returns and Exchanges,medium,"product,feature,tech support"
3,query about smart home system integration feat...,"dear customer support team,\n\ni hope this mes...",Request,Billing and Payments,low,"billing,payment,account,documentation,feedback"
4,inquiry regarding invoice details,"dear support team,\n\ni hope this message reac...",Problem,Sales and Pre-Sales,medium,"product,feature,feedback,tech support"


In [ ]:
print("Original Shape:", df.shape)
df.head()

Original Shape: (28587, 6)


,subject,Body,type,queue,priority,Tags
0,subject,"dear support team, i would like to report a se...",Incident,Technical Support,high,"security,outage,disruption,data breach"
1,major security incident,"dear customer support team,\n\ni am writing to...",Incident,Technical Support,high,"account,disruption,outage,it,tech support"
2,account disruption,"dear customer support team,\n\ni hope this mes...",Request,Returns and Exchanges,medium,"product,feature,tech support"
3,query about smart home system integration feat...,"dear customer support team,\n\ni hope this mes...",Request,Billing and Payments,low,"billing,payment,account,documentation,feedback"
4,inquiry regarding invoice details,"dear support team,\n\ni hope this message reac...",Problem,Sales and Pre-Sales,medium,"product,feature,feedback,tech support"


In [ ]:
df['subject'] = df['subject'].fillna('')   # replace missing subject
df['text'] = df['subject'] + " " + df['Body']

In [ ]:
df = df[['text', 'type', 'Tags', 'queue']]
df = df.dropna()

In [ ]:
print("After Selecting Columns:", df.shape)
df.head()

After Selecting Columns: (28587, 4)


,text,type,Tags,queue
0,"subject dear support team, i would like to rep...",Incident,"security,outage,disruption,data breach",Technical Support
1,major security incident dear customer support ...,Incident,"account,disruption,outage,it,tech support",Technical Support
2,"account disruption dear customer support team,...",Request,"product,feature,tech support",Returns and Exchanges
3,query about smart home system integration feat...,Request,"billing,payment,account,documentation,feedback",Billing and Payments
4,inquiry regarding invoice details dear support...,Problem,"product,feature,feedback,tech support",Sales and Pre-Sales


In [ ]:
stop_words = set(stopwords.words('english'))
lemma = WordNetLemmatizer()

In [ ]:
def clean_text(text):
    text = re.sub(r'http\S+|www\S+|https\S+', '', text)       # remove urls
    text = re.sub(r'<.*?>', '', text)                         # remove html tags
    text = re.sub(r'\s+', ' ', text).strip()                  # remove extra spaces

    # remove stopwords and lemmatize
    words = text.split()
    words = [lemma.lemmatize(w) for w in words if w not in stop_words]

    return " ".join(words)

In [ ]:
df['clean_text'] = df['text'].apply(clean_text)
df.head(10)

,text,type,Tags,queue,clean_text
0,"subject dear support team, i would like to rep...",Incident,"security,outage,disruption,data breach",Technical Support,"subject dear support team, would like report s..."
1,major security incident dear customer support ...,Incident,"account,disruption,outage,it,tech support",Technical Support,major security incident dear customer support ...
2,"account disruption dear customer support team,...",Request,"product,feature,tech support",Returns and Exchanges,"account disruption dear customer support team,..."
3,query about smart home system integration feat...,Request,"billing,payment,account,documentation,feedback",Billing and Payments,query smart home system integration feature de...
4,inquiry regarding invoice details dear support...,Problem,"product,feature,feedback,tech support",Sales and Pre-Sales,inquiry regarding invoice detail dear support ...
5,question about marketing agency software compa...,Request,"feature,product,documentation,feedback",Technical Support,question marketing agency software compatibili...
6,"feature query dear customer support team,\n\ni...",Incident,"outage,disruption,performance,it,tech support",Service Outages and Maintenance,"feature query dear customer support team,\n\ni..."
7,"system interruptions dear support team,\n\ni a...",Incident,"network,hardware,performance,bug,compatibility",Technical Support,"system interruption dear support team,\n\ni re..."
8,connectivity problems with printer on macbook ...,Request,"documentation,feedback,it,tech support",Technical Support,connectivity problem printer macbook pro dear ...
9,request for detailed information on the platfo...,Request,"disruption,outage,recovery,support",Service Outages and Maintenance,request detailed information platform's system...


LABEL ENCODING (USING QUEUE)

In [ ]:
if "queue" not in df.columns:
    raise Exception("Column 'type' not found in dataset!")

df["queue"] = df["queue"].fillna("Unknown").astype(str)

label_encoder = LabelEncoder()
df["label"] = label_encoder.fit_transform(df["queue"])

print("Classes:", list(label_encoder.classes_))

Classes: ['Billing and Payments', 'Customer Service', 'General Inquiry', 'Human Resources', 'IT Support', 'Product Support', 'Returns and Exchanges', 'Sales and Pre-Sales', 'Service Outages and Maintenance', 'Technical Support']


Train-Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    df["clean_text"], df["label"], test_size=0.20, random_state=42, stratify=df["label"]
)

TF-IDF Vectorization

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
tfidf = TfidfVectorizer(
    max_features=50000,
    ngram_range=(1,2),
    min_df=3,
    stop_words="english"
)

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

Training Models

In [ ]:
import pandas as pd
import numpy as np
import re
import nltk
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, classification_report
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

In [ ]:
models = {
    "Naive Bayes": MultinomialNB(),
    "Linear SVM": LinearSVC(),
    "Logistic Regression": LogisticRegression(max_iter=2000)
}

for name, model in models.items():
    model.fit(X_train_tfidf, y_train)
    preds = model.predict(X_test_tfidf)

    print("\n==============================")
    print(f"MODEL → {name}")
    print("==============================")
    print("Accuracy:", accuracy_score(y_test, preds))
    print("F1 (macro):", f1_score(y_test, preds, average="macro"))
    print("\nClassification Report:")
    print(classification_report(y_test, preds, target_names=label_encoder.classes_))


MODEL → Naive Bayes
Accuracy: 0.40328786288912205
F1 (macro): 0.17891967173541343

Classification Report:
                                 precision    recall  f1-score   support

           Billing and Payments       0.94      0.44      0.60       558
               Customer Service       0.40      0.32      0.36       854
                General Inquiry       0.00      0.00      0.00        81
                Human Resources       0.00      0.00      0.00       115
                     IT Support       0.43      0.00      0.01       687
                Product Support       0.48      0.22      0.30      1050
          Returns and Exchanges       0.00      0.00      0.00       287
            Sales and Pre-Sales       0.00      0.00      0.00       184
Service Outages and Maintenance       0.00      0.00      0.00       230
              Technical Support       0.36      0.93      0.52      1672

                       accuracy                           0.40      5718
               

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))



MODEL → Linear SVM
Accuracy: 0.6516264428121721
F1 (macro): 0.6376359621095724

Classification Report:
                                 precision    recall  f1-score   support

           Billing and Payments       0.82      0.79      0.80       558
               Customer Service       0.62      0.62      0.62       854
                General Inquiry       0.94      0.41      0.57        81
                Human Resources       0.86      0.43      0.57       115
                     IT Support       0.62      0.55      0.59       687
                Product Support       0.59      0.59      0.59      1050
          Returns and Exchanges       0.73      0.50      0.59       287
            Sales and Pre-Sales       0.83      0.46      0.59       184
Service Outages and Maintenance       0.78      0.76      0.77       230
              Technical Support       0.62      0.76      0.68      1672

                       accuracy                           0.65      5718
                  

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Training Model with TF-IDF + ML Model

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, accuracy_score
import pickle

In [ ]:
#Preparining Data
X = df['clean_text']
y = df['queue']   # single label for each ticket

In [ ]:
#Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [ ]:
#TF-IDF Vectorization
tfidf = TfidfVectorizer(max_features=5000, stop_words='english')
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

In [ ]:
#Training with linear SVM
svm_model = LinearSVC()
svm_model.fit(X_train_tfidf, y_train)

LinearSVC()

Training and Evaluating the model with RNN

In [ ]:
#Evaluating the model
preds = svm_model.predict(X_test_tfidf)

print("Accuracy:", accuracy_score(y_test, preds))
print(classification_report(y_test, preds))

Accuracy: 0.49860090940888424
                                 precision    recall  f1-score   support

           Billing and Payments       0.69      0.70      0.70       558
               Customer Service       0.47      0.42      0.44       854
                General Inquiry       0.71      0.21      0.32        81
                Human Resources       0.53      0.22      0.31       115
                     IT Support       0.41      0.34      0.37       687
                Product Support       0.42      0.42      0.42      1050
          Returns and Exchanges       0.51      0.29      0.37       287
            Sales and Pre-Sales       0.49      0.26      0.34       184
Service Outages and Maintenance       0.64      0.63      0.64       230
              Technical Support       0.50      0.67      0.57      1672

                       accuracy                           0.50      5718
                      macro avg       0.54      0.41      0.45      5718
                   

In [ ]:
import pandas as pd
import numpy as np
import re
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense, Dropout
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.callbacks import EarlyStopping

In [ ]:
#Preparing data
X = df['clean_text'].astype(str)
y = df['type'].astype(str)

In [ ]:
#label encoding
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)
y_categorical = to_categorical(y_encoded)
num_classes = len(label_encoder.classes_)

In [ ]:

#Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y_categorical, test_size=0.2, random_state=42, stratify=y
)

In [ ]:
#Tokenization and Padding
#setting parameters
vocab_size = 20000
max_len = 200
#Fitting Tokenizer
tokenizer = Tokenizer(num_words=vocab_size, oov_token="<OOV>")
tokenizer.fit_on_texts(X_train)

X_train_seq = tokenizer.texts_to_sequences(X_train)
X_test_seq = tokenizer.texts_to_sequences(X_test)

X_train_pad = pad_sequences(X_train_seq, maxlen=max_len, padding='post')
X_test_pad = pad_sequences(X_test_seq, maxlen=max_len, padding='post')

In [ ]:
#Building RNN/LSTM Model
model = Sequential([
    Embedding(vocab_size, 128, input_length=max_len),
    LSTM(128, return_sequences=False),
    Dropout(0.3),
    Dense(64, activation='relu'),
    Dropout(0.2),
    Dense(num_classes, activation='softmax')
])

model.compile(
    loss='categorical_crossentropy',
    optimizer='adam',
    metrics=['accuracy']
)

model.summary()

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [ ]:
#Traininf the model
history = model.fit(
    X_train_pad, y_train,
    validation_split=0.2,
    epochs=5,
    batch_size=128,
    callbacks=[EarlyStopping(monitor='val_loss', patience=2)]
)

Epoch 1/5
143/143 ━━━━━━━━━━━━━━━━━━━━ 160s 1s/step - accuracy: 0.3873 - loss: 1.3045 - val_accuracy: 0.3983 - val_loss: 1.2839
Epoch 2/5
143/143 ━━━━━━━━━━━━━━━━━━━━ 120s 838ms/step - accuracy: 0.3999 - loss: 1.2966 - val_accuracy: 0.3983 - val_loss: 1.2831
Epoch 3/5
143/143 ━━━━━━━━━━━━━━━━━━━━ 139s 819ms/step - accuracy: 0.4022 - loss: 1.2906 - val_accuracy: 0.3983 - val_loss: 1.2840
Epoch 4/5
143/143 ━━━━━━━━━━━━━━━━━━━━ 119s 827ms/step - accuracy: 0.3990 - loss: 1.2894 - val_accuracy: 0.3983 - val_loss: 1.2831
Epoch 5/5
143/143 ━━━━━━━━━━━━━━━━━━━━ 117s 822ms/step - accuracy: 0.4011 - loss: 1.2929 - val_accuracy: 0.3983 - val_loss: 1.2826


In [ ]:
#Model Evaluation
loss, acc = model.evaluate(X_test_pad, y_test)
print(f"Test Accuracy: {acc}")

179/179 ━━━━━━━━━━━━━━━━━━━━ 19s 108ms/step - accuracy: 0.4060 - loss: 1.2871
Test Accuracy: 0.4010143280029297


## We got low score for RNN, that's why choosing SVM over RNN

In [ ]:
#Saving the TF-IDF Vectorizer & SVM Model
with open("queue_svm_tfidf.pkl", "wb") as f:
    pickle.dump(tfidf, f)

with open("queue_svm_ticket_model.pkl", "wb") as f:
    pickle.dump(svm_model, f)

In [ ]:
#Loading saved model for prediction
with open("queue_svm_tfidf.pkl", "rb") as f:
    loaded_tfidf = pickle.load(f)

with open("queue_svm_ticket_model.pkl", "rb") as f:
    loaded_svm = pickle.load(f)

In [ ]:
#predicting new type for new ticket
def predict_ticket_type(text):
    clean = text.lower()
    clean = re.sub(r"[^a-zA-Z\s]", " ", clean)
    clean = re.sub(r"\s+", " ", clean).strip()

    vec = loaded_tfidf.transform([clean])
    pred = loaded_svm.predict(vec)[0]
    return pred

In [ ]:
ticket = "dear support team, i would like to report a serious security incident currently affecting multiple components of our infrastructure. affected devices include projectors, displays, and storage solutions on cloud platforms. the reason for this assumption is that the incident represents a potential data breach related to a cyberattack, posing a significant risk to sensitive information and the ongoing business operations of our organization. our initial investigations have revealed unusual activity and anomalies in the devices. despite implementing our standard remediation and containment measures, the threat has not yet been fully eliminated."
print("Predicted Type:", predict_ticket_type(ticket))

Predicted Type: Technical Support


In [ ]:
ticket = "Payment failed during checkout, I was charged twice."
print("Predicted Type:", predict_ticket_type(ticket))

Predicted Type: Billing and Payments


TYPE BASED CATEGORIZATION

In [ ]:
if "type" not in df.columns:
    raise Exception("Column 'type' not found in dataset!")

df["type"] = df["type"].fillna("Unknown").astype(str)

label_encoder = LabelEncoder()
df["label"] = label_encoder.fit_transform(df["type"])

print("Classes:", list(label_encoder.classes_))

Classes: ['Change', 'Incident', 'Problem', 'Request']


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    df["clean_text"], df["label"], test_size=0.20, random_state=42, stratify=df["label"]
)

TF-IDF Vectorization

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
tfidf = TfidfVectorizer(
    max_features=50000,
    ngram_range=(1,2),
    min_df=3,
    stop_words="english"
)

X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

In [ ]:
models = {
    "Naive Bayes": MultinomialNB(),
    "Linear SVM": LinearSVC(),
    "Logistic Regression": LogisticRegression(max_iter=2000)
}

for name, model in models.items():
    model.fit(X_train_tfidf, y_train)
    preds = model.predict(X_test_tfidf)

    print("\n==============================")
    print(f"MODEL → {name}")
    print("==============================")
    print("Accuracy:", accuracy_score(y_test, preds))
    print("F1 (macro):", f1_score(y_test, preds, average="macro"))
    print("\nClassification Report:")
    print(classification_report(y_test, preds, target_names=label_encoder.classes_))


MODEL → Naive Bayes
Accuracy: 0.7597061909758657
F1 (macro): 0.6568016492489469

Classification Report:
              precision    recall  f1-score   support

      Change       1.00      0.60      0.75       584
    Incident       0.66      0.99      0.79      2293
     Problem       0.95      0.08      0.15      1203
     Request       0.89      0.99      0.94      1638

    accuracy                           0.76      5718
   macro avg       0.88      0.67      0.66      5718
weighted avg       0.82      0.76      0.69      5718


MODEL → Linear SVM
Accuracy: 0.8642882126617698
F1 (macro): 0.8659839704865275

Classification Report:
              precision    recall  f1-score   support

      Change       0.97      0.95      0.96       584
    Incident       0.82      0.88      0.85      2293
     Problem       0.74      0.62      0.68      1203
     Request       0.98      0.99      0.98      1638

    accuracy                           0.86      5718
   macro avg       0.87      0

Training Model with TF-IDF + ML Model

In [ ]:
#Preparining Data
X = df['clean_text']
y = df['type']   # single label for each ticket

In [ ]:
#Train-Test Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

In [ ]:
#TF-IDF Vectorization
tfidf = TfidfVectorizer(max_features=5000, stop_words='english')
X_train_tfidf = tfidf.fit_transform(X_train)
X_test_tfidf = tfidf.transform(X_test)

In [ ]:
#Training with linear SVM
svm_model = LinearSVC()
svm_model.fit(X_train_tfidf, y_train)

LinearSVC()

In [ ]:
#Evaluating the model
preds = svm_model.predict(X_test_tfidf)

print("Accuracy:", accuracy_score(y_test, preds))
print(classification_report(y_test, preds))

Accuracy: 0.8151451556488283
              precision    recall  f1-score   support

      Change       0.94      0.94      0.94       584
    Incident       0.76      0.84      0.80      2293
     Problem       0.62      0.48      0.54      1203
     Request       0.97      0.98      0.98      1638

    accuracy                           0.82      5718
   macro avg       0.82      0.81      0.81      5718
weighted avg       0.81      0.82      0.81      5718



In [ ]:
#Saving the TF-IDF Vectorizer & SVM Model
with open("type_svm_tfidf.pkl", "wb") as f:
    pickle.dump(tfidf, f)

with open("type_svm_ticket_model.pkl", "wb") as f:
    pickle.dump(svm_model, f)

In [ ]:
#Loading saved model for prediction
with open("type_svm_tfidf.pkl", "rb") as f:
    loaded_tfidf = pickle.load(f)

with open("type_svm_ticket_model.pkl", "rb") as f:
    loaded_svm = pickle.load(f)

In [ ]:
#predicting new type for new ticket
def predict_ticket_type(text):
    clean = text.lower()
    clean = re.sub(r"[^a-zA-Z\s]", " ", clean)
    clean = re.sub(r"\s+", " ", clean).strip()

    vec = loaded_tfidf.transform([clean])
    pred = loaded_svm.predict(vec)[0]
    return pred

In [ ]:
ticket = "dear support team, i would like to report a serious security incident currently affecting multiple components of our infrastructure. affected devices include projectors, displays, and storage solutions on cloud platforms. the reason for this assumption is that the incident represents a potential data breach related to a cyberattack, posing a significant risk to sensitive information and the ongoing business operations of our organization. our initial investigations have revealed unusual activity and anomalies in the devices. despite implementing our standard remediation and containment measures, the threat has not yet been fully eliminated."
print("Predicted Type:", predict_ticket_type(ticket))

Predicted Type: Incident


In [ ]:
ticket = "Payment failed during checkout, I was charged twice."
print("Predicted Type:", predict_ticket_type(ticket))

Predicted Type: Problem


Loading Required Models for Predecting Type and Queues

In [ ]:
import pickle
import re

# Load TYPE model
with open("type_svm_tfidf.pkl", "rb") as f:
    type_vectorizer = pickle.load(f)

with open("type_svm_ticket_model.pkl", "rb") as f:
    type_model = pickle.load(f)

# Load QUEUE model
with open("queue_svm_tfidf.pkl", "rb") as f:
    queue_vectorizer = pickle.load(f)

with open("queue_svm_ticket_model.pkl", "rb") as f:
    queue_model = pickle.load(f)

In [ ]:
#Text Cleaning Function
def clean_text(text):
    text = text.lower()
    text = re.sub(r"[^a-zA-Z\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text


In [ ]:
#Final Function
def predict_ticket_type_and_queue(ticket_subject):
    """
    Input  : ticket subject (string)
    Output : dictionary with predicted type and queue
    """

    # Clean input
    cleaned_text = clean_text(ticket_subject)

    # -------- TYPE PREDICTION --------
    type_vec = type_vectorizer.transform([cleaned_text])
    predicted_type = type_model.predict(type_vec)[0]

    # -------- QUEUE PREDICTION --------
    queue_vec = queue_vectorizer.transform([cleaned_text])
    predicted_queue = queue_model.predict(queue_vec)[0]

    return {
        "Predicted_Type": predicted_type,
        "Predicted_Queue": predicted_queue
    }


In [ ]:
ticket = "Unable to login after password reset"

result = predict_ticket_type_and_queue(ticket)

print("Type :", result["Predicted_Type"])
print("Queue:", result["Predicted_Queue"])

Type : Incident
Queue: Technical Support
